In [1]:
import torch
import os

# --- FIX: FORCE INITIALIZATION FOR TORCH._DYNAMO ---
try:
    import torch._dynamo
    import torch._dynamo.decorators
except (ImportError, AttributeError):
    pass
# --------------------------------------------------

import gymnasium as gym
from gymnasium import spaces
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.results_plotter import load_results, ts2xy
import matplotlib.pyplot as plt
import pandas as pd
import random
from collections import deque

class MahimahiTraceManager:
    """
    Mengelola file trace Mahimahi (.log).
    Mengonversi timestamp (ms) menjadi Throughput (Mbps) dengan amplifikasi realistis.
    """
    def __init__(self, folder_path="traces_folder"):
        self.traces = []
        PACKET_SIZE_BITS = 1500 * 8 
        
        if os.path.exists(folder_path):
            files = [f for f in os.listdir(folder_path) if f.endswith('.log')]
            files.sort()
            
            for file in files:
                path = os.path.join(folder_path, file)
                try:
                    with open(path, 'r') as f:
                        timestamps_ms = [float(line.strip()) for line in f if line.strip()]
                        
                        if timestamps_ms:
                            throughput_mbps = []
                            current_sec = 0
                            packet_count = 0
                            
                            for ts in timestamps_ms:
                                sec = int(ts / 1000)
                                while current_sec < sec:
                                    mbps = (packet_count * PACKET_SIZE_BITS) / 1_000_000
                                    throughput_mbps.append(mbps)
                                    packet_count = 0
                                    current_sec += 1
                                packet_count += 1
                            
                            throughput_mbps.append((packet_count * PACKET_SIZE_BITS) / 1_000_000)
                            
                            # Scaling (Max ~12 Mbps untuk memicu kompetisi kualitas)
                            max_tp = max(throughput_mbps) if throughput_mbps else 1
                            scale_factor = 12.0 / max_tp if max_tp > 0 else 1.0
                            
                            scaled_mbps = [max(0.1, tp * scale_factor) for tp in throughput_mbps]
                            # Pre-smoothing trace dasar
                            smoothed = pd.Series(scaled_mbps).rolling(window=3, min_periods=1).mean().tolist()
                            
                            self.traces.append({
                                "name": file,
                                "data": smoothed
                            })
                except Exception as e:
                    print(f"Gagal memproses file {file}: {e}")
        
        if not self.traces:
            print("⚠️ traces_folder tidak ditemukan. Gunakan fallback.")
            self.traces.append({"name": "synth", "data": [8.0] * 500})

        self.active_trace = None
        self.ptr = 0

    def select_random_trace(self):
        self.active_trace = random.choice(self.traces)
        self.ptr = random.randint(0, max(0, len(self.active_trace["data"]) - 110))
        return self.active_trace["name"]

    def set_trace_index(self, idx):
        self.active_trace = self.traces[idx % len(self.traces)]
        self.ptr = 0
        return self.active_trace["name"]

    def get_next_bandwidth(self):
        val = self.active_trace["data"][self.ptr]
        self.ptr = (self.ptr + 1) % len(self.active_trace["data"])
        return val

class HybridStreamingEnv(gym.Env):
    """
    Hybrid ABR Architecture: RL + Dynamic Controller with Volatility Awareness.
    """
    def __init__(self, trace_manager):
        super(HybridStreamingEnv, self).__init__()
        self.trace_manager = trace_manager
        self.bitrates = [0.5, 2.5, 8.0] 
        
        # Observation Space: [Buffer, Mean_TP, Last_Exec, Buffer_Safety, Volatility]
        self.observation_space = spaces.Box(
            low=np.array([0.0, 0.0, 0.0, 0.0, 0.0]), 
            high=np.array([1.0, 1.0, 1.0, 1.0, 1.0]), 
            dtype=np.float32
        )
        
        self.action_space = spaces.Discrete(3)
        self.state = None 
        self.max_steps = 100
        self.current_step = 0
        
        # History panjang untuk kalkulasi statistik Volatility
        self.tp_history = deque(maxlen=10) 
        
        # Parameters
        self.LOW_BUFFER_THRESHOLD = 5.0 
        self.SAFE_MARGIN = 1.5
        self.UPGRADE_CONFIRM_STEPS = 2
        self.PANIC_BUFFER_THRESHOLD = 3.0

    def _get_normalized_obs(self):
        # State internal: [buffer, mean_tp, last_exec, _, volatility]
        buffer, mean_tp, last_exec, _, volatility = self.state
        
        norm_buffer = np.clip(buffer / 30.0, 0.0, 1.0)           
        norm_tp = np.clip(mean_tp / 10.0, 0.0, 1.0) 
        norm_exec = float(last_exec) / 2.0                   
        
        # Buffer Safety
        safety = max(0, buffer - self.LOW_BUFFER_THRESHOLD)
        norm_safety = np.clip(safety / 15.0, 0.0, 1.0)
        
        # Volatility (CV) di-clip agar tidak merusak gradien Neural Network
        norm_volatility = np.clip(volatility, 0.0, 1.0)
        
        return np.array([norm_buffer, norm_tp, norm_exec, norm_safety, norm_volatility], dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if options and "trace_idx" in options:
            trace_name = self.trace_manager.set_trace_index(options["trace_idx"])
        else:
            trace_name = self.trace_manager.select_random_trace()
            
        initial_tp = self.trace_manager.get_next_bandwidth()
        self.tp_history.clear()
        for _ in range(self.tp_history.maxlen):
            self.tp_history.append(initial_tp)
            
        # Initial State: [buffer, mean_tp, last_exec, 0.0, volatility]
        self.state = np.array([15.0, initial_tp, 1.0, 0.0, 0.0], dtype=np.float32)
        self.current_step = 0
        return self._get_normalized_obs(), {"trace": trace_name}

    def step(self, action):
        if isinstance(action, np.ndarray):
            action = action.item()
        
        target_idx = int(action)
        buffer, _, current_idx, _, _ = self.state
        current_idx = int(current_idx)
        
        # 1. Update Throughput & Hitung Volatility
        raw_tp = self.trace_manager.get_next_bandwidth()
        self.tp_history.append(raw_tp)
        
        mean_tp = np.mean(self.tp_history)
        std_tp = np.std(self.tp_history)
        volatility = std_tp / (mean_tp + 1e-6) # Coefficient of Variation
        
        # Headroom pakai Mean agar stabil
        headroom = mean_tp - self.bitrates[current_idx]
        
        # --- LAYER 2: DYNAMIC CONTROLLER (Volatility-Aware) ---
        executed_idx = current_idx

        # Rate Limiter (Max 1 level)
        if abs(target_idx - current_idx) > 1:
            target_idx = current_idx + np.sign(target_idx - current_idx)

        # LOGIKA SAFETY DINAMIS: Makin liar jaringan, makin tinggi buffer yang dibutuhkan
        dynamic_buffer_req = self.LOW_BUFFER_THRESHOLD + (volatility * 5.0)

        # 2. Panic Mode
        if buffer < self.PANIC_BUFFER_THRESHOLD:
            executed_idx = min(target_idx, current_idx - 1)
            executed_idx = max(0, executed_idx)
            self.upgrade_confidence_counter = 0
            
        # 3. Defensive Downgrade (Diperketat dengan Volatility)
        elif headroom < -2.0 and buffer < dynamic_buffer_req:
            executed_idx = max(0, current_idx - 1)
            self.upgrade_confidence_counter = 0

        # 4. Upgrade Hysteresis (Diperketat dengan Volatility)
        elif target_idx > current_idx:
            # Upgrade hanya jika margin kencang DAN buffer di atas standar dinamis
            if headroom > self.SAFE_MARGIN and buffer > dynamic_buffer_req:
                self.upgrade_confidence_counter += 1
            else:
                self.upgrade_confidence_counter = 0
            
            if self.upgrade_confidence_counter >= self.UPGRADE_CONFIRM_STEPS:
                executed_idx = current_idx + 1
                self.upgrade_confidence_counter = 0
            else:
                executed_idx = current_idx 
                
        # 5. Normal / Explicit Downgrade
        else:
            executed_idx = target_idx
            self.upgrade_confidence_counter = 0

        executed_idx = np.clip(executed_idx, 0, 2)

        # Eksekusi Fisik (DASH Simulation)
        chosen_bitrate = self.bitrates[executed_idx]
        seg_duration = 5.0
        download_time = (chosen_bitrate * seg_duration / (raw_tp + 0.1)) 
        stalling = max(0, download_time - buffer)
        
        new_buffer = max(0, buffer - download_time) + seg_duration
        new_buffer = min(new_buffer, 30.0)
        
        # Reward Strategy
        reward = float(chosen_bitrate * 1.0)
        if stalling > 0:
            reward -= float(stalling * 50.0 + 20.0) 
        
        if executed_idx != current_idx:
            # Penalti switch adaptif
            if buffer > 15.0: reward -= 5.0
            else: reward -= 1.0

        # Update Internal State
        self.state = np.array([new_buffer, mean_tp, float(executed_idx), 0.0, volatility], dtype=np.float32)
        self.current_step += 1
        done = self.current_step >= self.max_steps
        
        info = {
            "raw_tp": raw_tp, 
            "buffer": new_buffer,
            "executed_bitrate": chosen_bitrate,
            "volatility": volatility,
            "target_idx": target_idx,
            "executed_idx": executed_idx
        }
        
        return self._get_normalized_obs(), reward, done, False, info

def run_experiment():
    log_dir = "./rl_logs/"
    os.makedirs(log_dir, exist_ok=True)
    
    tm = MahimahiTraceManager(folder_path="traces_folder/mahimahi")
    if not tm.traces: return
        
    env = Monitor(HybridStreamingEnv(tm), log_dir)
    
    print("⏳ Memulai pelatihan 'Volatility-Aware Hybrid ABR' (300,000 langkah)...")
    model = PPO("MlpPolicy", env, verbose=1, 
                learning_rate=0.00025,  
                ent_coef=0.05, 
                n_steps=2048,
                batch_size=64)         
    
    model.learn(total_timesteps=800000)
    model.save("volatility_hybrid_ndn_model")
    print("✅ Pelatihan selesai.")

    # EVALUASI 8 FILE
    for i in range(len(tm.traces)):
        obs, info = env.reset(options={"trace_idx": i})
        trace_name = info["trace"]
        history = []
        for _ in range(100):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, _, info_step = env.step(action)
            action_val = action.item() if isinstance(action, np.ndarray) else int(action)
            history.append({
                'TP': info_step['raw_tp'],
                'Volatility': info_step['volatility'],
                'Buffer': info_step['buffer'], 
                'Exec_Bitrate': info_step['executed_bitrate']
            })
            
        df = pd.DataFrame(history)
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
        
        # Grafik Bitrate + Throughput
        ax1.plot(df.index, df['TP'], label='Raw TP', color='blue', alpha=0.15)
        ax1.step(df.index, df['Exec_Bitrate'], label='Executed Bitrate', color='red', linewidth=2, where='post')
        ax1.set_title(f"Volatility-Aware: {trace_name}")
        ax1.set_ylabel("Mbps")
        ax1.legend()
        
        # Grafik Buffer + Volatility
        ax2.fill_between(df.index, df['Buffer'], color='green', alpha=0.15, label='Buffer Level')
        ax2_vol = ax2.twinx()
        ax2_vol.plot(df.index, df['Volatility'], label='Volatility (CV)', color='purple', linestyle=':', alpha=0.6)
        ax2_vol.set_ylabel("Volatility Scale")
        ax2.axhline(y=5, color='orange', linestyle='--', label='Base Safe')
        ax2.set_xlabel("Segmen")
        ax2.legend(loc='upper right')
        
        plt.tight_layout()
        plt.savefig(os.path.join(log_dir, f"vol_hybrid_{trace_name}.png"))
        plt.close()

if __name__ == "__main__":
    run_experiment()

/root/.pyenv/versions/vevn/lib/python3.11/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/root/.pyenv/versions/vevn/lib/python3.11/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


⏳ Memulai pelatihan 'Volatility-Aware Hybrid ABR' (300,000 langkah)...
Using cpu device
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 100      |
|    ep_rew_mean     | -159     |
| time/              |          |
|    fps             | 3557     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 100         |
|    ep_rew_mean          | -185        |
| time/                   |             |
|    fps                  | 2792        |
|    iterations           | 2           |
|    time_elapsed         | 1           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.011071072 |
|    clip_fraction        | 0.0837      |
|    clip_range           | 0.2  